In [ ]:
Problem: Design Twitter
Difficulty: Medium
Link: https://leetcode.com/problems/design-twitter/
Tags: NeetCode 150

Example:
Twitter(), postTweet(1,5), getNewsFeed(1) -> [5], follow(1,2), postTweet(2,6), getNewsFeed(1) -> [6,5], unfollow(1,2), getNewsFeed(1) -> [5]

Constraints:
- tweetId values are unique
- getNewsFeed returns at most 10 recent tweet ids


In [ ]:
import heapq

class Twitter:
    def __init__(self):
        #DAG of user following edge key: node/ user value: list of out going following edges, including seeing their own post, which is self edge.
        self.follow_graph = dict() # dictionary of sets of user follow graph
        self.followed_graph = dict()
        
        self.user_tweets = dict() # source of truth for user tweets, dictionary of lists
        self.user_feed = dict() #dictionary of user feed heaps
        self.time = 0 

        # there is not time stamp system, but since there's a time stream, either can run a deque or heap or heap scan on top of deque, or should there be feed be user? 
        # does it make sense to sync heaps on follow, and post tweet
        # However during unfollow might have to dig deeper, however heap first 10 is log(10) time. so we can just store everything
        # so now should there be a universal heap or heap per user synced?
        # technically when we follow we just have to merge two heaps symmetrically, however during unfollow un merging is hard.
        # heap is definitely ordered in time, but it's usually by value. if we were doing dequeue we wouldn't need value since that's the default assumption. that things are streamed in time order. 

        # Definitely need dag for follows store, can use maps since faster add and remove and search. 

        # Case 1 when heap is used to store exactly newsfeed for the user
        # since merging requires resolving tweets between user and follower it's just inserting follower in to user feed, single directional following matters for feed.
        # unfollow requires pruning tweets from user feed which are from the unfollowed user.
        
        # case 2 when heap is used to store the news feed for everyone
        # technically it's easier to just store the edges and then search through the heap if it has tuple and poster id and pick the last 10 which are within users set of following.
        # since if unfollow follow happens often merging is expensive, right now unfollow follow is O(1). however searching through the heap will be worst case O(N),
        # but average case should be log(N), since the global time order is maintained in this heap --> but that would just require a single deque and doesn't required the complexity of maintaining a heap. 

        # case 3 when heap is only run time merging of user tweets queue during getting news feed. --> does not seem good since news feed probably happens more often that follow unfollow, thus constantly maintaing the feed heap is better.
 
        # let me solve it one way first, using case 1 which benefits from heap (case 2 doesn't) if i added a time and merging during sort can help. Case 3 doesn't seem practical for news feed follow unfollow ratio. then tracer bullet pattern if it isn't good optimize.

        # Case 1.
        # assumption that the news feed is always maintained as the heap, and i just need to check heap top 10 every time news feed check. 

        # shit following nodes needs to be updated on post, this action is more often than follow unfollow action, but we can maintain two sets of the inverse if needed also assuming fast action we use dictionary instead of edge lists.


    def initialize_user(self, userId):
        if userId not in self.follow_graph:
            self.follow_graph[userId] = {userId}
            self.followed_graph[userId] = {userId}
            user_heap = []
            heapq.heapify(user_heap)
            self.user_feed[userId] = user_heap
            self.user_tweets[userId] = []

    def postTweet(self, userId: int, tweetId: int) -> None:
        # Composes a new tweet with ID tweetId by the user userId. Each call to this function will be made with a unique tweetId.
        self.initialize_user(userId)
        for user in self.followed_graph[]

        



    def getNewsFeed(self, userId: int):
        # Retrieves the 10 most recent tweet IDs in the user's news feed.
        # Each item in the news feed must be posted by users who the user followed or by the user themself. Tweets must be ordered from most recent to least recent.
        self.initialize_user(userId)
        return heapq.nlargest(10, self.user_feed[userId])



    def follow(self, followerId: int, followeeId: int) -> None:
        # The user with ID followerId started following the user with ID followeeId.
        self.initialize_user(followerId)
        self.initialize_user(followeeId)
        self.follow_graph[followerId].add(followeeId)
        self.followed_graph[followeeId].add(followerId)


        # it seems like merging heaps might have something to do with underlying list, and perhaps 
        # it is useful to study inster delete get random o1 that i do it right first


    def unfollow(self, followerId: int, followeeId: int) -> None:
        #assuming user exist, can upgrade to try except later but thats not part of leetcode.
        # if neither exist then equivalent to noOp can add exceptions later no need for now in this leetcode exercise
        self.follow_graph[followerId].remove(followeeId)
        self.followed_graph[followeeId].remove(followerId)
        # prune from heap
        for _ in range(len(self.heap)):
            if self.heap:
                






In [ ]:
def test(solution):
    cases = [
        ((["postTweet", "getNewsFeed", "follow", "postTweet", "getNewsFeed", "unfollow", "getNewsFeed"],
          [[1, 5], [1], [1, 2], [2, 6], [1], [1, 2], [1]]),
         [None, [5], None, None, [6, 5], None, [5]]),
        ((["postTweet", "postTweet", "follow", "getNewsFeed", "unfollow", "getNewsFeed"],
          [[1, 10], [2, 20], [1, 2], [1], [1, 2], [1]]),
         [None, None, None, [20, 10], None, [10]]),
    ]
    for i, (args, expected) in enumerate(cases, 1):
        got = solution(*args)
        assert got == expected, f'case {i}: expected {expected}, got {got}'



In [ ]:
def current_solution(ops, params):
    obj = Twitter()
    out = []
    for op, arg in zip(ops, params):
        out.append(getattr(obj, op)(*arg))
    return out

result = "PASS (No solution provided to execute)"
print(result)
# When Twitter is runnable, replace the two lines above with:
# test(current_solution)
# print("PASS")



## Hint-Only Thinking Questions

- If you keep a per-user newsfeed heap, what invariant must always stay true after every `postTweet`, `follow`, and `unfollow`?
    ans. The heap is always be up to date with the follow graph propagated, the freshness if using global counter and heap should be maintained

- Which operation becomes more expensive when you maintain that invariant eagerly, and which becomes cheaper?
    ans. If i maintain that invariant of updating: technically posting and following and unfollowing get more expensive. while newsfeed is cheaper.
    however since posting is single insertion it should be standard cost to insert into all the followee's news feed.

- How often is `getNewsFeed` called relative to writes in your expected workload, and how should that ratio influence where you pay the cost?
    ans. getnewsfeed is called most often but in the question there's no specification: ```All the tweets have unique IDs.
At most 3 * 104 calls will be made to postTweet, getNewsFeed, follow, and unfollow.``` is all it says.

- On `follow`, do you need to import all historical tweets immediately, or can you justify a lazier boundary without breaking correctness?


- On `unfollow`, how will you guarantee removed users’ tweets stop appearing without scanning too much stale data?
- If heaps can contain stale entries, what quick validity check can you do at pop-time to preserve correctness?
- What is the maximum heap growth per user over time under your design, and does that create memory drift?
- Can you derive tight big-O for each API under both designs: eager updates on writes vs centralized merge in `getNewsFeed`?
- Which design is simpler to prove correct under edge cases like self-follow rules, repeated follow/unfollow, and duplicate posts?
- If you cap feed size to 10, where can bounded-size structures reduce work most effectively?
- What timestamp ordering guarantee do you rely on, and does your design ever risk violating global recency?
- If you had to explain your choice in one sentence as a read-heavy vs write-heavy tradeoff, what would that sentence be?
